# ENGR 1451/2451 — Weekly Assignment 8
**10 points**

This assignment focuses on three goals from Week 7 that were not emphasized in LE07: **experimental design**, **how sample size changes the precision of a sample mean**, and **using confidence intervals to plan a study**.

**Due:** See Canvas.

Complete the work directly in this notebook. Show your Python code and give brief written explanations where requested. Submit the completed notebook as directed in Canvas.

## What to hand in

Work this assignment with the **course tutor** in the class project. That is the AI help to use on this assignment, and it keeps the record for you — you do not write one yourself.

At the end of each working session, type **"make my record"** in the chat. Copy the block it produces and paste it into the **Session records** cell near the bottom of this notebook.

You do not have to finish in one sitting. To pick up later, start a new chat in the same project, say which assignment you are on and which problem you stopped at, and keep going. That session gives you its own block, which you paste under the previous one.

Records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. The code and the written answers still have to be yours.

## LE8-1. Study design and causal evidence (3 pts)

For each study, classify it as a **randomized controlled experiment**, **observational study**, or **historical-control study**.

**a.** An engineer has 60 comparable metal specimens. The specimens are randomly assigned to either a new heat treatment or the standard heat treatment. All specimens are tested on the same fatigue-testing machine using the same procedure. 

**b.** A manufacturer introduces a redesigned bearing this year and compares its failure rate with failure records from the previous bearing design five years ago. 

**c.** A hospital compares outcomes for patients who chose to receive a therapy with outcomes for patients who chose not to receive it. 
Then answer:

1. Which study gives the strongest basis for a **causal** conclusion?
2. What does random assignment help protect against?
3. Suppose the person measuring fatigue life knows which treatment each specimen received. What additional design feature could reduce measurement bias? 

Keep your response to about 5–6 sentences total.

**Answer:**

**a.**  This experiment is a **randomized control experiment** because the specimens are randomly assigned to experimental (new treatment) or control (standard treatment) groups.

**b.**  This is a **historical-control study**. There is an exprimental group (new bearing design) being compared to a control group (old bearing design) which consists of historical data.

**c.**  This is an example of an **observational study**, as there is no random assignment to groups and participation in the experimental group (therapy) is voluntary.

1. The randomized controlled experiment will give the strongest basis for a causal conclusion because it features random assignment to groups (this helps to protect against specimen to specimen variation) and keeps the largest number of independent variables (ex. time, shape, testing machine, etc) constant between the two group.
2. Random assignment helps to protect against cofounding.
3. Making the researcher/person measuring fatigue life unaware of which treatment each specimen received could help reduce measurmenet bias (known as a blinding). 

## LE8-2. Sample size and the precision of a mean (4 pts)

The file `./data/nist_aluminum_fatigue_21000psi.csv` contains fatigue-life measurements for **101 aluminum specimens** tested at a maximum stress of 21,000 psi. Fatigue life is reported in **thousand cycles**.

Treat these 101 measurements as a **sample from a broader population of comparable specimens manufactured and tested under the same conditions**. The population mean $\mu$ and population SD $\sigma$ are unknown.

The setup cell below calculates the sample mean and sample SD. We will use the observed sample SD as an estimate of $\sigma$ when thinking about the precision of future sample means.

In [49]:
import numpy as np
import pandas as pd
import scipy.stats as stats

pd.set_option("display.precision", 4)

fatigue = pd.read_csv("./data/nist_aluminum_fatigue_21000psi.csv")
life = fatigue["fatigue_life_thousand_cycles"].to_numpy()

n_observed = len(life)
sample_mean = life.mean()
sample_sd = life.std(ddof=1)

print(f"n = {n_observed}")
print(f"sample mean = {sample_mean:.1f} thousand cycles")
print(f"sample SD = {sample_sd:.1f} thousand cycles")


n = 101
sample mean = 1400.9 thousand cycles
sample SD = 391.3 thousand cycles


### A. How sample size changes the SE (2.5 pts)

Use the current sample SD as a planning estimate of $\sigma$.

For planned future sample sizes

```python
planned_n = np.array([25, 100, 400])
```

calculate the estimated standard error

$
\widehat{SE}(\bar X)=\frac{s}{\sqrt{n}}.
$

Create a DataFrame with columns:

- `n`
- `estimated_se`

Report SE in **thousand cycles**.

Then answer:

1. What happens to the SE each time the sample size is multiplied by 4?
2. Does increasing the sample size change the population mean being estimated, or only the precision of the estimate?

**Answer:** 
1. The SE is halved every time the sample size, n, is multiplied by 4. This is because the SE is inversely proportional to the square root of n, and $\sqrt{4n} = 2\sqrt{n}$.
2. Increasing the sample size only changes the precision of the estimate for the population mean (parameter).

In [50]:
planned_n = np.array([25, 100, 400])
# Your code here

# Calculate the estimated SE for each planned sample size.
estimated_se = sample_sd/(planned_n ** 0.5)

# Create and display a DataFrame with columns n and estimated_se.
estimated_se_table = (pd.DataFrame([planned_n,estimated_se],index = ["n","estimated_se"])).transpose() #transposen to get n and estimated_se as columns
estimated_se_table

,n,estimated_se
0,25.0,78.2648
1,100.0,39.1324
2,400.0,19.5662


### B. Normal approximation for a future sample mean (1.5 pts)

Assume future measurements are independent and come from the same stable production process. Use the current sample SD $s$ as an estimate of $\sigma$.

For future samples of size $n=25$ and $n=100$, approximate the probability that the sample mean will be within **100 thousand cycles** of the true population mean:

$
P\left(|\bar X-\mu|\leq100\right).
$

For each sample size:

1. estimate $SE(\bar X)=s/\sqrt{n}$;
2. convert 100 thousand cycles to standard units,
   $
   z=\frac{100}{SE};
   $
3. use the normal curve:
   ```python
   stats.norm.cdf(z) - stats.norm.cdf(-z)
   ```

Display the two probabilities as percentages.

In one sentence, explain why the larger sample size gives a higher probability even though $\mu$ is unknown.

**Answer:** The larger sample gives a higher probability that sample mean will be within 100 thousand cycles of the true population mean because the larger sample has a smaller SE, thus increasing the z-value and decreasing the probability that the sample mean will fall outside of +- 100 thousand cycles. The population mean is not needed for this calculation because $\bar X-\mu$ gives the difference between the population mean and the sample mean, so inputting 100 (for a difference of 100 thousand cycles) is all that is neccesary.

In [51]:
sample_sizes = np.array([25, 100])

# Your code here

# Calculate estimated SE for each sample size.
standard_errors = sample_sd/(sample_sizes ** 0.5)

# Convert +/-100 thousand cycles to z units.
cycle_tolerance = 100 #in units of thousand cycles
z = (cycle_tolerance)/standard_errors

# Calculate P(-z <= Z <= z) for each sample size.
interval_probabilities = stats.norm.cdf(z) - stats.norm.cdf(-z)

# Display the results as percentages.
percentage_comparison_table = pd.DataFrame([sample_sizes,standard_errors,z,interval_probabilities*100],index = ["n","SE","z","P(-z <= Z <= z) [%]"]).transpose()
percentage_comparison_table



,n,SE,z,P(-z <= Z <= z) [%]
0,25.0,78.2648,1.2777,79.8649
1,100.0,39.1324,2.5554,98.9394


## LE8-3. Planning sample size for a confidence interval (3 pts)

Suppose the fatigue-life measurements above are a **pilot sample** for a future study. Use `sample_sd` as a planning estimate of the unknown population SD.

The goal is a **95% confidence interval for the population mean** with a specified margin of error.

For a planned sample size $n$,

$
\text{margin of error} $
$=
t^*_{n-1}\frac{s}{\sqrt{n}}.
$

This extends the confidence-interval calculation from LE07: here the question is not *what interval do we get from one sample?* but rather *how many measurements should we plan to collect?*

In [52]:
candidate_n = np.arange(5, 501)

t_star = stats.t.ppf(0.975, df=candidate_n - 1)
estimated_se = sample_sd / np.sqrt(candidate_n)
margin_of_error = t_star * estimated_se

### A. Find a sample size using Student's $t$ (2 pts)

Using the arrays above, find the **smallest** sample size for which the estimated 95% margin of error is no more than:

1. **100 thousand cycles**
2. **50 thousand cycles**

Boolean indexing is one way to do this:

```python
candidate_n[margin_of_error <= target][0]
```

Display both required sample sizes.

Then answer: approximately what happens to the required sample size when the target margin of error is cut in half?

**Answer:** Cutting the margin of error in half results in a ~4x increase in required sample size.

In [53]:
# Your code here

# Find the smallest candidate_n meeting each target margin of error.
tolerances = np.array([100, 50])

def req_samp_size_calc(tolerances, ME, value_range):
    req_sample_size = np.zeros(len(tolerances))
    for pos, tolerance in enumerate(tolerances):
        req_sample_size[pos] = value_range[ME <= tolerance][0]

    return req_sample_size

req_sample_size = req_samp_size_calc(tolerances, margin_of_error, candidate_n)

required_sampe_size_table = pd.DataFrame([tolerances,req_sample_size],index=["Desired Margin of Error","Required Sample Size"]).transpose()
required_sampe_size_table

,Desired Margin of Error,Required Sample Size
0,100.0,62.0
1,50.0,238.0


### B. Compare with the large-sample normal shortcut (1 pt)

For a target margin of error of **100 thousand cycles**, repeat the calculation using

```python
z_star = stats.norm.ppf(0.975)
normal_margin_of_error = z_star * sample_sd / np.sqrt(candidate_n)
```

Find the smallest sample size from the normal approximation.

In 1–2 sentences, compare this result with the Student's $t$ result from part A. Why is the $t$-based requirement slightly more conservative when $\sigma$ is unknown?

**Answer:** The z-based required sample size (59) is smaller than the t-based required sample size (62) for a margin of error of 100 thousand cycles. This is because the z distribution assumes a known standard deviation whereas the t disribution does not, meaning that the t distribution is generally wider and has a higher $t^*$ multiplier for the margin of error (which in this case is shown by needing a higher $n$ to compensate).

In [54]:
z_star = stats.norm.ppf(0.975)

# Your code here

# Calculate the normal-approximation margin of error or every value in candidate_n.
normal_margin_of_error = z_star * sample_sd / np.sqrt(candidate_n)

# Find the smallest sample size giving margin of error <= 100.
tolerances_new = np.array([100])
req_sample_size_new = req_samp_size_calc(tolerances_new, normal_margin_of_error, candidate_n)
    
required_sampe_size_table_new = pd.DataFrame([tolerances_new,req_sample_size_new],index=["Desired Margin of Error","Required Sample Size"]).transpose()
required_sampe_size_table_new

,Desired Margin of Error,Required Sample Size
0,100.0,59.0


## Session records

At the end of each working session, type **"make my record"** in the chat and paste the block it produces into the cell below. If you work across several sessions, paste each new block underneath the previous one.

Assignments are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. They are read to find out which steps are hardest, and used to improve the course.

<!-- Paste your session record blocks in this cell, one after another. -->

*(session records go here)*
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA8",
  "export_n": 1,
  "session_date": "2026-10-25",
  "messages": [
    {"i": 1, "who": "student", "text": "Can you help check my work on part 1? \n\n```markdown\n**a.**  This experiment is a **randomized control experiment** because the specimens are randomly assigned to experimental (new treatment) or control (standard treatment) groups.\n\n**b.**  This is a **historical-control study**. There is an exprimental group (new bearing design) being compared to a control group (old bearing design) which consists of historical data.\n\n**c.**  This is an example of an **observational study**, as there is no random assignment to groups and participation in the experimental group (therapy) is voluntary.\n\n1. The randomized controlled experiment will give the strongest basis for a causal conclusion because it features random assignment to groups and keeps the largest number of independent variables (ex. time, shape, testing machine, etc) constant between the two group.\n2. Random assignment helps to protect against cofounding.\n3. Making the researcher/person measuring fatigue life unaware of which treatment each specimen received could help reduce measurmenet bias (known as a double-blind study). \n```"},
    {"i": 2, "who": "tutor", "text": "Your three classifications and your answers to questions 1, 2, and 3 all land on the right ideas. Two of your explanations deserve a closer look.\n\n**Question 1.** Your choice is right, but check your reason. You say the experiment \"keeps the largest number of variables constant between the two groups.\" Which part of the design holds the machine and procedure constant, and what does random assignment do to the other factors, such as differences among the specimens themselves? Is \"constant\" the right word for that second job?\n\n**Question 3.** The feature you named is right, but you attached \"double-blind\" to it. Think about who is actually in this study: the subjects are metal specimens. Who would need to be kept unaware of the treatment, and does anyone else exist in this design who could be? [trimmed]\n\n**Length.** The prompt asks for about 5-6 sentences total, and your response runs a bit longer. [trimmed]\n\nWant to revise those two points and paste the new version, or move on to LE8-2?"},
    {"i": 3, "who": "student", "text": "```markdown\n**Answer:**\n\n**a.**  This experiment is a **randomized control experiment** because the specimens are randomly assigned to experimental (new treatment) or control (standard treatment) groups.\n\n**b.**  This is a **historical-control study**. There is an exprimental group (new bearing design) being compared to a control group (old bearing design) which consists of historical data.\n\n**c.**  This is an example of an **observational study**, as there is no random assignment to groups and participation in the experimental group (therapy) is voluntary.\n\n1. The randomized controlled experiment will give the strongest basis for a causal conclusion because it features random assignment to groups (this helps to protect against specimen to specimen variation) and keeps the largest number of independent variables (ex. time, shape, testing machine, etc) constant between the two group.\n2. Random assignment helps to protect against cofounding.\n3. Making the researcher/person measuring fatigue life unaware of which treatment each specimen received could help reduce measurmenet bias (known as a blinding). \n```"},
    {"i": 4, "who": "tutor", "text": "You tightened the question 3 wording nicely. \"Blinding\" fits a design whose subjects are metal specimens, since the only person who could be kept unaware is the one measuring. [trimmed] **Question 1.** ... Is it accurate to say random assignment makes those factors \"constant\"? Or does it do something different to them? ... **Length.** ... One small spelling note: \"confounding\" ... and \"measurement\" ... Do you want to revise the question 1 sentence, or move on to LE8-2?"},
    {"i": 5, "who": "student", "text": "Moving on"},
    {"i": 6, "who": "tutor", "text": "Sounds good. Your LE8-1 answers are in good shape, and you handled the revision on question 3 well.\n\nHere is LE8-2.", "served": ["a8_le8_2"]},
    {"i": 7, "who": "student", "text": "```python\nplanned_n = np.array([25, 100, 400])\n# Your code here\n\n# Calculate the estimated SE for each planned sample size.\nestimated_se = sample_sd/(planned_n ** 0.5)\n\n# Create and display a DataFrame with columns n and estimated_se.\nestimated_se_table = (pd.DataFrame([planned_n,estimated_se],index = [\"n\",\"estimated_se\"])).transpose() #transposen to get n and estimated_se as columns\nestimated_se_table\n```\n\n\n```markdown\n**Answer:** \n1. The SE is halved every time the sample size, n, is multiplied by 4. This is because the SE is inversley proportional to the square root of n, and $\\sqrt{4n} = 2\\sqrt{n}$.\n2. Increasing the sample size only changes the precision of the estimate for the population mean (parameter).\n```", "code": true},
    {"i": 8, "who": "tutor", "text": "Your SEs come from `sample_sd` and `planned_n`, so they use the current sample SD, and the table has the two columns the prompt names. Your answers to both questions match the facts, and your reason for the first one is well argued. [trimmed] Two small things: the `n` column may display as floats because of how the DataFrame was built; the name `estimated_se` is reassigned by the LE8-3 starter cell. Also, \"inversley\" ... Want to move on to part B, or revisit anything here?"},
    {"i": 9, "who": "student", "text": "Can I use stats.norm.cdf on a list of values?"},
    {"i": 10, "who": "tutor", "text": "Yes. `stats.norm.cdf` accepts a single number, a list, or a NumPy array, and it returns the area to the left of each value. [trimmed] A quick demonstration on made-up data: `z_values = [-1, 0, 1.5]` then `stats.norm.cdf(z_values)`. [trimmed] Does that answer what you were wondering, or is there a specific line you're trying to write?"},
    {"i": 11, "who": "student", "text": "Am I doing anything wrong in my code? I am getting 100 for my % values.\n\n```python\nsample_sizes = np.array([25, 100])\n\n# Your code here\n\n# Calculate estimated SE for each sample size.\nstandard_errors = sample_sd/(sample_sizes ** 0.5)\n\n# Convert +/-100 thousand cycles to z units.\ncycle_tolerance = 100 #in units of thousand cycles\nz = (sample_mean + cycle_tolerance)/standard_errors\n\n# Calculate P(-z <= Z <= z) for each sample size.\ninterval_probabilities = stats.norm.cdf(z) - stats.norm.cdf(-z)\n\n# Display the results as percentages.\npercentage_comparison_table = pd.DataFrame([sample_sizes,standard_errors,interval_probabilities*100],index = [\"n\",\"SE\",\"P(-z <= Z <= z) [%]\"]).transpose()\npercentage_comparison_table\n\n\n```", "code": true},
    {"i": 12, "who": "tutor", "text": "Your SE line, the area expression, and the percentage conversion are all fine, and the table layout is clear. The problem is in the z line. Look at what you divide by the SE: `sample_mean + cycle_tolerance`. [trimmed] Ask yourself what the 100 in the prompt represents. The probability is P(|X̄ − μ| ≤ 100), so 100 is a distance ... [trimmed] Try changing that one line and rerun the cell."},
    {"i": 13, "who": "student", "text": "```python\nsample_sizes = np.array([25, 100])\n\n# Your code here\n\n# Calculate estimated SE for each sample size.\nstandard_errors = sample_sd/(sample_sizes ** 0.5)\n\n# Convert +/-100 thousand cycles to z units.\ncycle_tolerance = 100 #in units of thousand cycles\nz = (cycle_tolerance)/standard_errors\n\n# Calculate P(-z <= Z <= z) for each sample size.\ninterval_probabilities = stats.norm.cdf(z) - stats.norm.cdf(-z)\n\n# Display the results as percentages.\npercentage_comparison_table = pd.DataFrame([sample_sizes,standard_errors,interval_probabilities*100],index = [\"n\",\"SE\",\"P(-z <= Z <= z) [%]\"]).transpose()\npercentage_comparison_table\n\n\n```", "code": true},
    {"i": 14, "who": "tutor", "text": "Your z line now measures what the prompt asks: the 100 thousand cycle distance divided by the SE, with nothing about where the mean sits. [trimmed] The one piece still missing for this part is the one-sentence explanation in the **Answer:** field, and the wording has to be yours. Two questions may help: What is the probability actually a probability *of*? ... What sets how widely the sample mean's possible values are spread around μ ...?"},
    {"i": 15, "who": "student", "text": "```markdown\n**Answer:** The larger sample gives a higher probability that the true population mean will be within 100 thousand cycles of the sample mean because the larger sample has a smaller SE, thus tightening the margin of error.\n```"},
    {"i": 16, "who": "tutor", "text": "You've got the key driver: the larger sample has a smaller SE. [trimmed] **What is the probability about?** ... Compare that with the expression in the prompt, P(|X̄ − μ| ≤ 100). Which of the two is the random quantity ...? **What does \"tightening the margin of error\" refer to?** ... The prompt also asks why this works \"even though μ is unknown.\" [trimmed]"},
    {"i": 17, "who": "student", "text": "```markdown\n**Answer:** The larger sample gives a higher probability that the true population mean will be within 100 thousand cycles of the sample mean because the larger sample has a smaller SE, thus increasing the z-value and decreasing the probability that the sample mean will fall outside of +- 100 thousand cycles.\n```"},
    {"i": 18, "who": "tutor", "text": "Your revision does two good things: it keeps the one-sentence length, and it adds the link between the smaller SE and a larger z. [trimmed] **The start of your sentence.** It still says the probability is that \"the true population mean will be within 100 thousand cycles of the sample mean.\" ... which one could land somewhere different each time a new sample is collected? **\"Even though μ is unknown.\"** ... Does either input require knowing μ? [trimmed]"},
    {"i": 19, "who": "student", "text": "```markdown\n**Answer:** The larger sample gives a higher probability that sample mean will be within 100 thousand cycles of the true population mean because the larger sample has a smaller SE, thus increasing the z-value and decreasing the probability that the sample mean will fall outside of +- 100 thousand cycles.\n```"},
    {"i": 20, "who": "tutor", "text": "Your sentence now describes the probability the way the prompt does ... One piece of the prompt remains open: it asks why this works \"even though μ is unknown.\" [trimmed] A small typo to fix before submitting: the \"+-\" ... Want to revise that last point, or move on to LE8-3?"},
    {"i": 21, "who": "student", "text": "```markdown\n**Answer:** The larger sample gives a higher probability that sample mean will be within 100 thousand cycles of the true population mean because the larger sample has a smaller SE, thus increasing the z-value and decreasing the probability that the sample mean will fall outside of +- 100 thousand cycles. The population mean is not needed for this calculation because $\\bar X-\\mu$ gives the difference between the population mean and the sample mean, so inputting 100 (for a difference of 100 thousand cycles) is all that is neccesary.\n```"},
    {"i": 22, "who": "tutor", "text": "Your new sentences name the right idea: the calculation works with a distance, 100, and the SE, so you never need μ's value. [trimmed] **Length.** The prompt asks for one sentence, and this is now two. ... **Why the distance alone is enough.** ... what makes the area between −z and z computable without μ? [trimmed] Small spelling fix: \"neccesary\" ..."},
    {"i": 23, "who": "student", "text": "What does candidate_n represent?"},
    {"i": 24, "who": "tutor", "text": "`candidate_n` is a NumPy array of every sample size the notebook is willing to consider for the future study: 5, 6, 7, and so on up to 500. It comes from `np.arange(5, 501)`, and the stop value is left out ... [trimmed] One distinction to keep in mind: a position in the array is not the same thing as the sample size stored there. Because the array starts at 5, the value at position k is k + 5. Is there a particular line in LE8-3 you're trying to connect this to?"},
    {"i": 25, "who": "student", "text": "Would solving for n algbraically be acceptable?"},
    {"i": 26, "who": "tutor", "text": "Not for LE8-3A. The prompt says \"Using the arrays above,\" so the graded method is searching `candidate_n` with the Boolean mask. [trimmed] In LE8-3A the multiplier is t*, and t* depends on n through df = n − 1, so no single algebraic step isolates n. [trimmed] Algebra can still be useful as a sanity check ... [trimmed]"},
    {"i": 27, "who": "student", "text": "Is there a way to clean up this code"},
    {"i": 28, "who": "tutor", "text": "Happy to look. Which cell do you mean? I don't see new code in your last message, so paste the one you're working on ... [trimmed] Which cell should I look at?"},
    {"i": 29, "who": "student", "text": "```python\n# Your code here\n\n# Find the smallest candidate_n meeting each target margin of error.\ntolerances = np.array([100, 50])\n\ni = 0\nreq_sample_size = np.array([1,2])\nfor tolerance in tolerances:\n    req_sample_size[i] = candidate_n[margin_of_error <= tolerance][0]\n    i=i+1\n\nprint(req_sample_size)\n\nrequired_sampe_size_table = pd.DataFrame([tolerances,req_sample_size],index=[\"Desired Margin of Error\",\"Required Sample Size\"]).transpose()\nrequired_sampe_size_table\n```", "code": true},
    {"i": 30, "who": "tutor", "text": "Your cell does what part A asks. Each target gets the printed pattern, `[0]` picks the first size that meets it, and both sizes are displayed in a table. [trimmed] Optional polish: the manual counter `i` (`enumerate` as an alternative), the placeholder array `np.array([1,2])`, and the typo `required_sampe_size_table`. [trimmed] The one piece still open for this part is the **Answer:** about what happens to the required sample size when the target margin of error is cut in half. ..."},
    {"i": 31, "who": "student", "text": "Is there a way to create a specified size of np array?"},
    {"i": 32, "who": "tutor", "text": "Yes. NumPy has several functions that create an array of a specified size ... `np.zeros(2)`, `np.ones(3)`, `np.full(4, 7)`, `np.empty(2)`, `np.zeros(2, dtype=int)` [trimmed] The type matters here: `np.zeros(2)` makes floats by default ... Does that cover what you were after, or is there a specific line you're trying to restructure?"},
    {"i": 33, "who": "student", "text": "```markdown\n**Answer:** Cutting the margin of error in half results in a ~4x increase in required sample size.\n```\n\n\n```python\n# Your code here\n\n# Find the smallest candidate_n meeting each target margin of error.\ntolerances = np.array([100, 50])\n\nreq_sample_size = np.zeros(len(tolerances))\nfor pos, tolerance in enumerate(tolerances):\n    req_sample_size[pos] = candidate_n[margin_of_error <= tolerance][0]\n\nprint(req_sample_size)\n\nrequired_sampe_size_table = pd.DataFrame([tolerances,req_sample_size],index=[\"Desired Margin of Error\",\"Required Sample Size\"]).transpose()\nrequired_sampe_size_table\n```", "code": true},
    {"i": 34, "who": "tutor", "text": "Your answer to the halving question matches what your two sizes show, and a rounded factor is a responsive way to state it. [trimmed] One effect of the change to `np.zeros(len(tolerances))`: that array holds floats by default ... `dtype=int` ... That completes LE8-3A. Do you want to continue to LE8-3B, or stop here?"},
    {"i": 35, "who": "student", "text": "```python\n# Your code here\n\n# Find the smallest candidate_n meeting each target margin of error.\ntolerances = np.array([100, 50])\n\ndef req_samp_size_calc(tolerances, ME, value_range):\n    req_sample_size = np.zeros(len(tolerances))\n    for pos, tolerance in enumerate(tolerances):\n        req_sample_size[pos] = value_range[ME <= tolerance][0]\n\n    return req_sample_size\n\nreq_sample_size = req_samp_size_calc(tolerances, margin_of_error, candidate_n)\n\nrequired_sampe_size_table = pd.DataFrame([tolerances,req_sample_size],index=[\"Desired Margin of Error\",\"Required Sample Size\"]).transpose()\nrequired_sampe_size_table\n```", "code": true},
    {"i": 36, "who": "tutor", "text": "Your function does the same job as the loop, and it takes the margin array and the size array as arguments, so you can reuse it in part B with `normal_margin_of_error`. [trimmed] Two small optional things: float sizes (`dtype=int`) and the typo in the table name. Your halving answer is already in the Answer field, so part A is complete. Do you want to go on to LE8-3B, or stop here?"},
    {"i": 37, "who": "student", "text": "```python\nz_star = stats.norm.ppf(0.975)\n\n# Your code here\n\n# Calculate the normal-approximation margin of error or every value in candidate_n.\nmargin_of_error_new = z_star * sample_sd / np.sqrt(candidate_n)\n\n# Find the smallest sample size giving margin of error <= 100.\ntolerances_new = np.array([100])\nreq_sample_size_new = req_samp_size_calc(tolerances_new, margin_of_error_new, candidate_n)\n    \nrequired_sampe_size_table = pd.DataFrame([tolerances_new,req_sample_size_new],index=[\"Desired Margin of Error\",\"Required Sample Size\"]).transpose()\nrequired_sampe_size_table\n```", "code": true},
    {"i": 38, "who": "tutor", "text": "Your cell follows the prompt for this part: the normal-approximation margin is computed over every value in `candidate_n`, the target is 100, and your function searches `candidate_n` ... [trimmed] A few small things: the prompt's name `normal_margin_of_error`, reuse of the table name, float sizes and the typo. The written part is still open: ... which size is larger and by how much? ... how the t multiplier at these sample sizes compares with z*."},
    {"i": 39, "who": "student", "text": "```python\nz_star = stats.norm.ppf(0.975)\n\n# Your code here\n\n# Calculate the normal-approximation margin of error or every value in candidate_n.\nnormal_margin_of_error = z_star * sample_sd / np.sqrt(candidate_n)\n\n# Find the smallest sample size giving margin of error <= 100.\ntolerances_new = np.array([100])\nreq_sample_size_new = req_samp_size_calc(tolerances_new, normal_margin_of_error, candidate_n)\n    \nrequired_sampe_size_table_new = pd.DataFrame([tolerances_new,req_sample_size_new],index=[\"Desired Margin of Error\",\"Required Sample Size\"]).transpose()\nrequired_sampe_size_table_new\n```", "code": true},
    {"i": 40, "who": "tutor", "text": "You renamed the array to `normal_margin_of_error`, as in the prompt, and gave the new table its own name, so part A's table is no longer overwritten. [trimmed] The open piece is the **Answer:** field for this part: one or two sentences comparing your normal-approximation size with the t-based size from part A ... Paste your sentences when you have them and I'll take a look."},
    {"i": 41, "who": "student", "text": "```markdown\n**Answer:** The z-based required sample size (59) is smaller than the t-based required sample size (62) for a margin of error of 100 thousand cycles because the z distribution assumes a known standard deviation whereas the t disribution does not, meaning that the t distribution is generally wider, requiring a larger sample size to get the same value as the z distribution.\n```"},
    {"i": 42, "who": "tutor", "text": "Your two sizes match what I get from the project data: 59 for the normal approximation and 62 for the t-based search. [trimmed] What does the wider t distribution do to the multiplier in the margin formula? ... \"to get the same value as the z distribution\" is vague. Same value of what? ... Spelling note: \"disribution\" ..."},
    {"i": 43, "who": "student", "text": "```markdown\n**Answer:** The z-based required sample size (59) is smaller than the t-based required sample size (62) for a margin of error of 100 thousand cycles. This is because the z distribution assumes a known standard deviation whereas the t disribution does not, meaning that the t distribution is generally wider and has a higher $t^*$ multiplier for the margin of error (which in this case is shown by needing a higher $n$ to compensate).\n```"},
    {"i": 44, "who": "tutor", "text": "Your revision now makes the link I was pointing at: the wider t distribution gives a larger multiplier, and a larger n is needed to compensate. [trimmed] You've now finished all five parts. Type \"make my record\" when you're ready ..."}
  ],
  "events": [
    {"i": 5, "part": "8-1", "type": "unstuck"},
    {"i": 7, "part": "8-2A", "type": "code"},
    {"i": 8, "part": "8-2A", "type": "part_pass"},
    {"i": 9, "part": "8-2B", "type": "explain"},
    {"i": 11, "part": "8-2B", "type": "code"},
    {"i": 13, "part": "8-2B", "type": "code"},
    {"i": 22, "part": "8-2B", "type": "part_pass"},
    {"i": 23, "part": "8-3A", "type": "explain"},
    {"i": 25, "part": "8-3A", "type": "explain"},
    {"i": 29, "part": "8-3A", "type": "code"},
    {"i": 31, "part": "8-3A", "type": "explain"},
    {"i": 33, "part": "8-3A", "type": "code"},
    {"i": 34, "part": "8-3A", "type": "part_pass"},
    {"i": 35, "part": "8-3A", "type": "code"},
    {"i": 37, "part": "8-3B", "type": "code"},
    {"i": 39, "part": "8-3B", "type": "code"},
    {"i": 44, "part": "8-3B", "type": "part_pass"}
  ],
  "parts": [
    {"part": "8-1", "exchanges": 3, "max_tier": 3, "passed": true},
    {"part": "8-2A", "exchanges": 1, "max_tier": 1, "passed": true},
    {"part": "8-2B", "exchanges": 7, "max_tier": 3, "passed": true},
    {"part": "8-3A", "exchanges": 7, "max_tier": 1, "passed": true},
    {"part": "8-3B", "exchanges": 4, "max_tier": 2, "passed": true}
  ],
  "tutor_assessment": [
    "The student stalled longest on the LE8-2B written sentence, which took four revisions. The sticking points were first treating μ rather than the future sample mean as the thing that varies, and then saying why not knowing μ does not block the calculation. An earlier snag was a z line that included sample_mean, which gave 100% for both sizes. The LE8-1 answer to question 1 also still blurs random assignment with holding factors constant.",
    "The student said \"Moving on\" once, at the end of LE8-1, while the question 1 wording and the length point were still open. That looks like the student's own choice rather than a failure of the tutoring. No release-valve walkthrough was used."
  ]
}

## Before you submit

1. Confirm that the provided dataset is available at `./data/nist_aluminum_fatigue_21000psi.csv`.
2. **Restart Kernel and Run All Cells** and confirm every completed cell runs without errors.
3. Confirm your printed values and written answers are visible in the saved file.
4. Confirm your session record blocks are pasted into the cell above.
5. Save the notebook, then upload the `.ipynb` to Canvas.